In [ ]:
# 08.0：RQAlpha 三组合正式比较与研究结论
from pathlib import Path
import json
import pickle

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")

RESULT_FILES = {
    "Q5_factor_high": (
        PROJECT_ROOT
        / "data"
        / "backtests"
        / "07_rqalpha_oos_q5_portable_result.pkl"
    ),
    "Q1_factor_low": (
        PROJECT_ROOT
        / "data"
        / "backtests"
        / "07_rqalpha_oos_q1_portable_result.pkl"
    ),
    "Equal_weight_eligible_universe": (
        PROJECT_ROOT
        / "data"
        / "backtests"
        / "07_rqalpha_oos_equal_weight_eligible_universe_portable_result.pkl"
    ),
}

TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures"
METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"

for directory in [TABLE_DIR, FIGURE_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

summary_rows = []
nav_series = {}

for portfolio_name, result_file in RESULT_FILES.items():
    assert result_file.exists(), f"未找到结果文件：{result_file}"

    with result_file.open("rb") as file:
        payload = pickle.load(file)

    assert payload["format"] == "rqalpha_analyser_portable_v1"

    summary = payload["summary"]
    summary_rows.append(
        {
            "portfolio": portfolio_name,
            "total_return": summary.get("total_returns"),
            "annualized_return": summary.get("annualized_returns"),
            "benchmark_total_return": summary.get("benchmark_total_returns"),
            "benchmark_annualized_return": summary.get(
                "benchmark_annualized_returns"
            ),
            "annualized_excess_return": summary.get(
                "annualized_excess_returns"
            ),
            "sharpe_ratio": summary.get("sharpe"),
            "information_ratio": summary.get("information_ratio"),
            "max_drawdown": summary.get("max_drawdown"),
        }
    )

    portfolio_payload = payload["portfolio"]
    portfolio_frame = pd.DataFrame(
        portfolio_payload["data"],
        columns=portfolio_payload["columns"],
    )

    date_candidates = [
        column
        for column in portfolio_frame.columns
        if column in ["index", "date", "datetime"]
    ]
    date_column = date_candidates[0] if date_candidates else portfolio_frame.columns[0]

    portfolio_frame[date_column] = pd.to_datetime(portfolio_frame[date_column])
    portfolio_frame = portfolio_frame.set_index(date_column).sort_index()

    nav_series[portfolio_name] = pd.to_numeric(
        portfolio_frame["unit_net_value"],
        errors="coerce",
    )

performance_summary = pd.DataFrame(summary_rows)

daily_nav = pd.concat(nav_series, axis=1).dropna()
daily_nav.index.name = "date"

relative_rows = []

for control_name in [
    "Q1_factor_low",
    "Equal_weight_eligible_universe",
]:
    relative_nav = (
        daily_nav["Q5_factor_high"] / daily_nav[control_name]
    )

    relative_total_return = relative_nav.iloc[-1] / relative_nav.iloc[0] - 1.0
    annualized_relative_return = (
        relative_nav.iloc[-1] / relative_nav.iloc[0]
    ) ** (252.0 / len(relative_nav)) - 1.0

    relative_rows.append(
        {
            "comparison": "Q5_factor_high_vs_{}".format(control_name),
            "relative_total_return": relative_total_return,
            "annualized_relative_return": annualized_relative_return,
        }
    )

relative_comparison = pd.DataFrame(relative_rows)

# 研究结论：不根据结果优化因子或权重。
decision_log = pd.DataFrame(
    [
        {
            "comparison": "Q5_vs_Q1",
            "finding": "Q5 的 RQAlpha 表现优于 Q1。",
            "decision": "positive_rank_ordering_evidence",
        },
        {
            "comparison": "Q5_vs_equal_weight_eligible_universe",
            "finding": "Q5 未稳定超过等权可投资券池。",
            "decision": "no_incremental_selection_alpha_confirmed",
        },
        {
            "comparison": "overall",
            "finding": (
                "当前等权合成对照版本保留为研究候选，"
                "但不能作为已确认的独立可交易 Alpha。"
            ),
            "decision": "candidate_not_confirmed",
        },
    ]
)

performance_path = (
    TABLE_DIR / "08_rqalpha_oos_three_portfolio_performance.csv"
)
relative_path = (
    TABLE_DIR / "08_rqalpha_oos_q5_relative_comparison.csv"
)
nav_path = (
    TABLE_DIR / "08_rqalpha_oos_three_portfolio_daily_nav.csv"
)
decision_path = (
    TABLE_DIR / "08_rqalpha_oos_strategy_research_decision.csv"
)
figure_path = (
    FIGURE_DIR / "08_rqalpha_oos_three_portfolio_comparison.png"
)
rules_path = (
    METADATA_DIR / "08_rqalpha_oos_strategy_comparison_rules.json"
)

performance_summary.to_csv(
    performance_path,
    index=False,
    encoding="utf-8-sig",
)
relative_comparison.to_csv(
    relative_path,
    index=False,
    encoding="utf-8-sig",
)
daily_nav.reset_index().to_csv(
    nav_path,
    index=False,
    encoding="utf-8-sig",
)
decision_log.to_csv(
    decision_path,
    index=False,
    encoding="utf-8-sig",
)

comparison_rules = {
    "engine": "rqalpha_plus",
    "scope": "out_of_sample_2023_2025_signals_with_final_exit",
    "comparison_portfolios": list(RESULT_FILES.keys()),
    "common_execution_settings": {
        "benchmark": "000906.XSHG",
        "matching_type": "current_bar",
        "price_limit": True,
        "volume_limit": True,
        "volume_percent": 0.05,
        "minimum_commission": 5.0,
    },
    "decision_rule": (
        "Q5 必须同时相对 Q1 与等权可投资券池呈现稳定增益，"
        "才可确认增量选股 Alpha。"
    ),
}

with rules_path.open("w", encoding="utf-8") as file:
    json.dump(comparison_rules, file, ensure_ascii=False, indent=2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

daily_nav.plot(ax=axes[0])
axes[0].set_title("RQAlpha Out-of-Sample Portfolio NAV")
axes[0].set_ylabel("Normalized NAV")
axes[0].grid(alpha=0.3)

relative_q1 = daily_nav["Q5_factor_high"] / daily_nav["Q1_factor_low"]
relative_universe = (
    daily_nav["Q5_factor_high"]
    / daily_nav["Equal_weight_eligible_universe"]
)

axes[1].plot(relative_q1, label="Q5 / Q1")
axes[1].plot(relative_universe, label="Q5 / Equal-weight universe")
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Q5 Relative NAV")
axes[1].set_ylabel("Relative NAV")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

print("===== RQAlpha 三组合比较完成 =====")
print("绩效表：", performance_path)
print("相对表现：", relative_path)
print("研究决策：", decision_path)
print("图表：", figure_path)

display(performance_summary)
display(relative_comparison)
display(decision_log)

In [ ]:
# 08.1：补充相对 CSI 800 的几何超额收益
benchmark_relative_rows = []

for portfolio_name, result_file in RESULT_FILES.items():
    with result_file.open("rb") as file:
        payload = pickle.load(file)

    benchmark_payload = payload["benchmark_portfolio"]

    benchmark_frame = pd.DataFrame(
        benchmark_payload["data"],
        columns=benchmark_payload["columns"],
    )

    date_candidates = [
        column
        for column in benchmark_frame.columns
        if column in ["index", "date", "datetime"]
    ]
    date_column = (
        date_candidates[0]
        if date_candidates
        else benchmark_frame.columns[0]
    )

    benchmark_frame[date_column] = pd.to_datetime(
        benchmark_frame[date_column]
    )

    benchmark_nav = (
        benchmark_frame
        .set_index(date_column)["unit_net_value"]
        .astype(float)
        .sort_index()
    )

    aligned_nav = pd.concat(
        [
            daily_nav[portfolio_name].rename("strategy_nav"),
            benchmark_nav.rename("benchmark_nav"),
        ],
        axis=1,
    ).dropna()

    relative_nav = (
        aligned_nav["strategy_nav"]
        / aligned_nav["benchmark_nav"]
    )

    benchmark_relative_rows.append(
        {
            "portfolio": portfolio_name,
            "relative_total_return_to_benchmark": (
                relative_nav.iloc[-1] / relative_nav.iloc[0] - 1.0
            ),
            "annualized_relative_return_to_benchmark": (
                relative_nav.iloc[-1] / relative_nav.iloc[0]
            ) ** (252.0 / len(relative_nav)) - 1.0,
        }
    )

benchmark_relative_summary = pd.DataFrame(
    benchmark_relative_rows
)

performance_summary = (
    performance_summary.drop(
        columns=[
            "relative_total_return_to_benchmark",
            "annualized_relative_return_to_benchmark",
        ],
        errors="ignore",
    )
    .merge(
        benchmark_relative_summary,
        on="portfolio",
        how="left",
    )
)

performance_summary.to_csv(
    performance_path,
    index=False,
    encoding="utf-8-sig",
)

print("===== 已补充相对 CSI 800 的几何超额收益 =====")
display(
    performance_summary[
        [
            "portfolio",
            "total_return",
            "annualized_return",
            "relative_total_return_to_benchmark",
            "annualized_relative_return_to_benchmark",
            "information_ratio",
            "max_drawdown",
        ]
    ]
)

In [ ]:
%%writefile docs/09_RESEARCH_PROTOCOL.md
# 盈利因子复现研究协议

## 研究定位

本项目是对 A 股盈利能力因子的可复现筛选研究，不是为展示历史收益而优化的策略项目。

研究参考盈利能力评价指标演进的公开研报思路，在 RiceQuant 点时数据环境中，对动态中证 800 股票池实施因子构造、横截面检验、样本外验证和执行约束回测。

所有结论以 2023-2025 样本外表现为主要依据。样本外结果已经被观察，不允许再据此优化组件权重、阈值、持有期或股票池规则。

## Alpha 假设

研究假设是：可持续盈利能力、较低盈利波动、较低杠杆扭曲和较高现金盈利质量，可能反映企业竞争优势、资本配置质量或市场对基本面持续性的定价滞后。

该假设并不等于已确认 Alpha。本项目要求因子不仅优于低分组，也应超过不按因子排序的等权可投资券池，才可被认定为增量选股 Alpha。

## 预测目标与策略映射

- 预测目标：股票横截面未来 20 个交易日开盘到开盘收益的排序；
- 信号日期：月末最后一个交易日；
- 收益标签：信号日后第一个交易日开盘买入、T+21 开盘卖出；
- 因子检验：IC、RankIC、五分组收益、Q5-Q1；
- 策略映射：因子最高 20% 股票等权做多；
- 重叠处理：按活跃持仓批次分配资金，最大同时活跃批次为 2。

## 数据与时间对齐

- 股票池：动态中证 800 成分股；
- 研究区间：2016-01 至 2025-12；
- 样本内：2016-2022；
- 样本外：2023-2025 信号，持有期延伸至 2026-02-02；
- 新股过滤：上市满 252 个交易日；
- 交易状态过滤：信号日 ST 与停牌股票不进入研究券池；
- 财务数据：RiceQuant 点时字段，采用 TTM/MRQ 口径；
- 横截面处理：MAD 去极值、CITIC 2019 一级行业中性化、对数市值中性化、z-score 标准化；
- FCFFIC：金融、非银金融、综合金融和房地产不参与测试。

## 因子范围

预先定义的研究范围包括：

1. 原始 TTM ROE；
2. 去杠杆 ROE；
3. 盈利稳定性；
4. 稳定去杠杆 ROE；
5. ROA、ROIC、RONOA 的严格口径或透明代理；
6. 重构 FCFFIC；
7. 基于研报四个底层子因子的综合盈利因子等权合成对照版本。

当前等权合成对照版本由稳定去杠杆 ROE、稳定 ROIC 代理、稳定 RONOA 代理和重构 FCFFIC 各 25% 构成，不根据样本外结果调权。该版本源于研报四个底层子因子的思路，但不是研报第 7.2 节原始两层权重公式。

## 样本外验证结果

RQAlpha 样本外正式回测使用相同的交易成本、最小佣金、涨跌停限制和成交量上限。

| 组合 | 累计收益 | 年化收益 | 相对 CSI 800 年化收益 | 信息比率 | 最大回撤 |
| --- | ---: | ---: | ---: | ---: | ---: |
| Q5 因子高组 | 28.012% | 8.874% | 3.732% | 0.582 | 24.398% |
| Q1 因子低组 | 23.108% | 7.419% | 2.301% | 0.234 | 29.916% |
| 等权可投资券池 | 28.489% | 9.013% | 3.782% | 0.689 | 25.685% |

Q5 相对 Q1 的累计收益为 4.117%，年化相对收益为 1.399%。

Q5 相对等权可投资券池的累计收益为 -0.141%，年化相对收益为 -0.049%。

## 当前研究决策

- Q5 优于 Q1，保留正向排序证据；
- Q5 未稳定超过等权可投资券池；
- 当前等权合成对照版本保留为候选研究对象；
- 不将其称为已确认的独立 Alpha，也不进入可直接部署策略库；
- 不基于当前样本外结果重新调参或拼接新权重。

最终状态：`candidate_not_confirmed`。

## 已识别的失败案例与执行局限

- 2023-2024 出现显著回撤，需做回撤区间与市场环境归因；
- RQAlpha 出现零股调仓导致的“下单数量为 0”重复提示；
- 最后一个交易日部分股票跌停，未能完成卖出，当前期末收益为按市值计价的组合净值；
- 当前未校准冲击成本与滑点；
- 当前未进行大资金容量压力测试；
- RQAlpha 原始 analyser 文件存在跨 pandas 版本读取限制，正式使用纯 Python 可移植 pkl 保存本地原始结果。

## 后续实验优先级

1. 对重叠收益标签使用 Newey-West/HAC 标准误；
2. 分 2023、2024、2025 三年检查 IC、RankIC、Q5-Q1 与相对等权券池表现；
3. 归因最大回撤、换手、交易成本和成交约束；
4. 进行期末强制清仓尾部敏感性检验；
5. 如项目目标转为实盘候选，再做滑点、成本和容量敏感性。

不会根据既有样本外结果优化因子定义或权重。

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(
    "/home/rice/notebook/a-share-profitability-factor-replication"
)

(PROJECT_ROOT / "docs").mkdir(parents=True, exist_ok=True)
os.chdir(str(PROJECT_ROOT))

print("当前工作目录：", Path.cwd())
print("docs 目录存在：", (PROJECT_ROOT / "docs").exists())

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(
    "/home/rice/notebook/a-share-profitability-factor-replication"
)
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"

keywords = [
    "integrated",
    "effectiveness",
    "rankic",
    "ic_",
    "quintile",
    "stable",
]

candidate_files = sorted(
    [
        path
        for path in TABLE_DIR.glob("*.csv")
        if any(keyword in path.name.lower() for keyword in keywords)
    ]
)

print("===== 09.1 可复用时间序列文件盘点 =====")

for file_path in candidate_files:
    preview = pd.read_csv(file_path, nrows=3)

    print("\n文件：", file_path.name)
    print("字段：", list(preview.columns))

In [ ]:
%%writefile lib/research_statistics.py
import numpy as np
import pandas as pd
import statsmodels.api as sm


def hac_mean_test(values, max_lags=1):
    """
    对任意一维数值序列的均值进行 Newey-West HAC 检验。

    参数
    ----
    values : array-like
        需要检验均值的时间序列。
    max_lags : int
        HAC 自相关滞后阶数。

    返回
    ----
    dict
        包含样本数、均值、HAC 标准误、t 统计量与 p 值。
    """
    series = pd.Series(values, dtype="float64").dropna()
    observation_count = len(series)

    if observation_count < 3:
        return {
            "observations": observation_count,
            "mean": np.nan,
            "hac_standard_error": np.nan,
            "hac_t_stat": np.nan,
            "hac_p_value": np.nan,
            "max_lags": max_lags,
        }

    effective_lags = min(max(int(max_lags), 0), observation_count - 1)

    fitted_model = sm.OLS(
        series.to_numpy(),
        np.ones((observation_count, 1)),
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": effective_lags},
    )

    return {
        "observations": observation_count,
        "mean": float(fitted_model.params[0]),
        "hac_standard_error": float(fitted_model.bse[0]),
        "hac_t_stat": float(fitted_model.tvalues[0]),
        "hac_p_value": float(fitted_model.pvalues[0]),
        "max_lags": effective_lags,
    }